# Learning and Adaptation

The Learning and Adaptation pattern enables agents to iteratively improve their outputs through an evaluate → select → mutate → repeat cycle. Inspired by evolutionary algorithms and systems like SICA and OpenEvolve, the agent uses an LLM to generate variants of a program, scores them, keeps the best, and repeats until a fitness threshold is reached or iterations are exhausted.

## Implementation with Flyte v2 + sandboxing

Evolving LLM-generated code is the canonical use case for Flyte's sandbox primitives. This notebook uses three:

- **`flyte.sandbox.create(...)`** — each candidate runs in its own fresh, ephemeral container, so untrusted generated code never executes in the orchestrator process.
- **`CodeModeAgent`** — for *adaptive* evaluation, the agent writes its own test cases and calls a tool to run them, instead of relying on a fixed suite.
- **`orchestrate_local(...)`** — the workflow-sandbox path: an LLM-emitted control-flow string is executed safely in the Monty runtime, calling only whitelisted functions.

#### OpenEvolve vs Flyte v2 + sandboxing

| Aspect | OpenEvolve | Flyte v2 + sandboxing |
|--------|-----------|------------------------|
| **Evolution loop** | `OpenEvolve(...)` + `await evolve.run(iterations=N)` | Explicit async loop inside `@env.task` |
| **Candidate execution** | Subprocess `evaluator.py` | `flyte.sandbox.create` — fresh container per candidate |
| **Test design** | Fixed `evaluator.py` | `CodeModeAgent` designs tests adaptively |
| **Dynamic control flow** | Library-internal | `orchestrate_local` — Monty workflow sandbox |
| **Observability** | Logs | `flyte.report` live tab + nested sandbox actions |
| **Secrets** | `os.environ` | `flyte.Secret` injected by cluster |

### 1. Install dependencies

In [ ]:
!uv pip install 'flyte[tui]' litellm anthropic pydantic-monty

### Start the devbox

If you haven't already, install the flyte package with the command above, then launch the local cluster:

In [ ]:
!flyte start devbox

### 2. Store your API key

In [ ]:
!flyte create secret ANTHROPIC_API_KEY --value sk-...

### 3. Import dependencies and configure the TaskEnvironment

In [ ]:
import json
import os
from dataclasses import dataclass, field
from datetime import timedelta

from anthropic import AsyncAnthropic
import flyte
import flyte.report
from flyte.sandbox import sandbox_environment

flyte.init_from_config()

_image = (
    flyte.Image.from_debian_base(name="evo-agent", python_version=(3, 12))
    .with_pip_packages("anthropic>=0.25.0")
)

# evolve_program orchestrates a sandbox, so it depends on the sandbox environment.
evo_env = flyte.TaskEnvironment(
    name="evo_agent",
    image=_image,
    resources=flyte.Resources(cpu="1", memory="2Gi"),
    secrets=[
        flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY"),
    ],
    depends_on=[sandbox_environment],
)

### 4. Define data models

OpenEvolve stores candidate programs as files and tracks metrics in a `Program` database. In Flyte v2, typed dataclasses replace file-based state — every generation is a serializable `ProgramVariant` that is visible as structured output in the UI.

In [7]:
@dataclass
class ProgramVariant:
    """A single candidate program and its evaluation score."""
    code: str = ""
    score: float = 0.0
    generation: int = 0
    mutation_notes: str = ""


@dataclass
class EvoResult:
    """Final output of the evolutionary coding agent."""
    task_description: str = ""
    best_code: str = ""
    best_score: float = 0.0
    generations: int = 0
    total_variants_tried: int = 0
    converged: bool = False


Sandbox-based evaluation (this section)

`_evaluate` no longer runs candidate code with a raw `exec()` in the task process. Instead it calls `eval_sandbox` — a `flyte.sandbox.create(...)` template — so **each candidate executes in its own fresh, isolated container**. The split of responsibilities:

| Step | Uses LLM? | Runs where |
|------|-----------|------------|
| `_mutate` | Yes (`AsyncAnthropic`) | task process (open-ended: LLM decides *how* to improve) |
| `_evaluate` | No | `flyte.sandbox.create` container (deterministic, isolated) |
| `_critique` | Yes (`AsyncAnthropic`) | task process (open-ended improvement direction) |

Section 8 then swaps the fixed test suite for a `CodeModeAgent` that designs tests itself, and section 9 shows the `orchestrate_local` workflow-sandbox path.

In [ ]:
MUTATION_SYSTEM = """\
You are an expert Python programmer. Given a function and feedback, produce an improved version.

Rules:
- Return ONLY the complete improved Python function, no explanation.
- Keep the same function signature.
- Make exactly one focused improvement per mutation.
- The function must be self-contained (no imports outside the function body)."""


@flyte.trace
async def _mutate(
    current_code: str,
    task_description: str,
    critique: str,
) -> ProgramVariant:
    """Ask the LLM to produce an improved variant. Replaces OpenEvolve's LLM sampler."""
    client = AsyncAnthropic()
    response = await client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        system=MUTATION_SYSTEM,
        messages=[{
            "role": "user",
            "content": (
                f"Task: {task_description}\n\n"
                f"Current code:\n```python\n{current_code}\n```\n\n"
                f"Critique / improvement direction: {critique}\n\n"
                "Produce the improved function:"
            ),
        }],
    )
    raw = response.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    return ProgramVariant(code=raw, mutation_notes=critique)


# ── Sandbox evaluator: each candidate runs in its OWN ephemeral container ──────
# flyte.sandbox.create builds the sandbox image once (cache="auto") and runs the
# snippet below with `code`, `fn_name`, `test_cases_json` injected as local
# variables (auto_io). Outputs are collected by name. The untrusted, LLM-generated
# candidate never executes in the orchestrator process.
_EVAL_CODE = """\
import json as _j
score = 0.0
error_msg = ""
passed = 0
total = 0
namespace = {}
try:
    exec(compile(code, "<candidate>", "exec"), namespace)
    fn = namespace.get(fn_name)
    if fn is None or not callable(fn):
        error_msg = "function not found"
    else:
        test_data = _j.loads(test_cases_json)
        total = len(test_data)
        for item in test_data:
            try:
                out = fn(*item["inputs"])
                if out == item["expected"]:
                    passed += 1
            except Exception as exc:
                if not error_msg:
                    error_msg = str(exc)
        score = passed / total if total > 0 else 0.0
except Exception as exc:
    error_msg = str(exc)
"""

eval_sandbox = flyte.sandbox.create(
    name="code-evaluator",
    code=_EVAL_CODE,
    inputs={"code": str, "fn_name": str, "test_cases_json": str},
    outputs={"score": float, "error_msg": str, "passed": int, "total": int},
    cache="disable",
)


@flyte.trace
async def _evaluate(
    variant: ProgramVariant,
    test_cases: list[tuple],
    fn_name: str,
) -> ProgramVariant:
    """Score a variant by running it in a fresh sandbox (flyte.sandbox.create).

    Replaces the earlier raw `exec()` evaluator: each candidate now runs in its own
    isolated, ephemeral container instead of the orchestrator's process.
    """
    import json as _j

    test_cases_json = _j.dumps([
        {"inputs": list(inp) if isinstance(inp, (tuple, list)) else [inp], "expected": exp}
        for inp, exp in test_cases
    ])
    score, error_msg, passed, total = await eval_sandbox.run.aio(
        code=variant.code, fn_name=fn_name, test_cases_json=test_cases_json,
    )
    return ProgramVariant(
        code=variant.code,
        score=float(score),
        generation=variant.generation,
        mutation_notes=(error_msg or f"{passed}/{total} tests passed") or variant.mutation_notes,
    )


@flyte.trace
async def _critique(
    variant: ProgramVariant,
    task_description: str,
    test_cases: list[tuple],
    fn_name: str,
) -> str:
    """Ask the LLM to critique the current best program and suggest a mutation direction."""
    client = AsyncAnthropic()
    failed = [
        f"  {inp} → expected {exp}"
        for inp, exp in test_cases
    ]
    response = await client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=256,
        system="You are a code reviewer. Suggest ONE specific improvement for a Python function.",
        messages=[{
            "role": "user",
            "content": (
                f"Task: {task_description}\n"
                f"Current score: {variant.score:.2f}\n"
                f"Code:\n```python\n{variant.code}\n```\n"
                f"Test cases (input → expected):\n" + "\n".join(failed) + "\n\n"
                "In one sentence, what should the next mutation focus on?"
            ),
        }],
    )
    return response.content[0].text.strip()

### 6. Define the evolutionary coding agent task

OpenEvolve's `await evolve.run(iterations=1000)` hides a generate → evaluate → select loop inside the library. In Flyte v2, this loop is explicit Python — the same logic, fully transparent:

1. **Mutate** the current best program with LLM guidance
2. **Evaluate** the variant against test cases
3. **Select** the better of old vs. new (elitist selection)
4. Update the live report and repeat

`@flyte.trace` on each helper means the task resumes from the last checkpoint on pod failure.

In [9]:
def _html_escape(text: str) -> str:
    return text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


@evo_env.task(
    retries=2,
    timeout=timedelta(minutes=15),
    cache=flyte.Cache(behavior="disable"),
    report=True,
)
async def evolve_program(
    task_description: str,
    initial_code: str,
    test_cases: list[list],   # list of [inputs, expected]; inputs is list for multi-arg
    fn_name: str,
    max_generations: int = 10,
    target_score: float = 1.0,
) -> EvoResult:
    """
    Evolutionary coding agent: iteratively improve a Python function.

    Replaces OpenEvolve's:
      evolve = OpenEvolve(initial_program_path, evaluation_file, config_path)
      best_program = await evolve.run(iterations=1000)

    The generate → evaluate → select loop is the same logic, now explicit.
    """
    # Convert nested lists to tuples for test_cases
    tc: list[tuple] = [(tuple(inp) if isinstance(inp, list) else inp, exp)
                       for inp, exp in test_cases]

    best = await _evaluate(
        ProgramVariant(code=initial_code, generation=0),
        tc, fn_name,
    )
    total_tried = 1
    report_rows: list[str] = []

    for gen in range(1, max_generations + 1):
        # Critique current best → get mutation direction
        critique = await _critique(best, task_description, tc, fn_name)

        # Generate a mutant
        mutant = await _mutate(best.code, task_description, critique)
        mutant = ProgramVariant(
            code=mutant.code,
            generation=gen,
            mutation_notes=mutant.mutation_notes,
        )

        # Evaluate the mutant
        mutant = await _evaluate(mutant, tc, fn_name)
        total_tried += 1

        # Elitist selection: keep the better program
        improved = mutant.score > best.score
        if improved:
            best = mutant

        color = "green" if improved else "gray"
        report_rows.append(
            f"<tr><td>{gen}</td>"
            f"<td style='color:{color}'>{mutant.score:.2f}</td>"
            f"<td>{best.score:.2f}</td>"
            f"<td>{_html_escape(critique[:80])}...</td></tr>"
        )

        await flyte.report.replace.aio(
            "<!DOCTYPE html><html><body>"
            f"<h2>Evolving: {_html_escape(task_description[:60])}</h2>"
            f"<p>Generation {gen} / {max_generations} | Best score: <strong>{best.score:.2f}</strong></p>"
            "<table border='1' cellpadding='4'>"
            "<tr><th>Gen</th><th>Mutant</th><th>Best</th><th>Critique</th></tr>"
            + "".join(report_rows)
            + "</table>"
            f"<h3>Current best code</h3><pre>{_html_escape(best.code)}</pre>"
            "</body></html>"
        )
        await flyte.report.flush.aio()

        if best.score >= target_score:
            break

    return EvoResult(
        task_description=task_description,
        best_code=best.code,
        best_score=best.score,
        generations=gen,
        total_variants_tried=total_tried,
        converged=best.score >= target_score,
    )

### 7. Run locally

In [ ]:
# Start with a broken implementation — the agent must fix it
INITIAL_CODE = """
def is_palindrome(s: str) -> bool:
    # placeholder — always returns False
    return False
""".strip()

TEST_CASES = [
    [["racecar"], True],
    [["hello"], False],
    [["level"], True],
    [["world"], False],
    [["madam"], True],
]

run = await flyte.run.aio(
    evolve_program,
    task_description="A function is_palindrome(s) that returns True if s reads the same forwards and backwards.",
    initial_code=INITIAL_CODE,
    test_cases=TEST_CASES,
    fn_name="is_palindrome",
    max_generations=8,
    target_score=1.0,
)
result: EvoResult = run.outputs()[0]

print(f"Converged: {result.converged}")
print(f"Best score: {result.best_score:.2f} after {result.generations} generations ({result.total_variants_tried} variants)")
print(f"\nBest code:\n{result.best_code}")

### Running remotely

The `report=True` flag enables a live HTML tab in the Flyte UI — watch each generation's score and critique update in real time as the agent evolves the program. The `EvoResult` dataclass makes the final best code and convergence metrics inspectable without parsing logs.

In [ ]:
run = await flyte.run.aio(
    evolve_program,
    task_description="A function binary_search(arr, target) returning the index of target in sorted arr, or -1 if not found.",
    initial_code="def binary_search(arr, target):\n    return -1  # placeholder",
    test_cases=[
        [[[1, 3, 5, 7, 9], 5], 2],
        [[[1, 3, 5, 7, 9], 1], 0],
        [[[1, 3, 5, 7, 9], 9], 4],
        [[[1, 3, 5, 7, 9], 4], -1],
    ],
    fn_name="binary_search",
    max_generations=12,
    target_score=1.0,
)
result = run.outputs()[0]
print(f"Score: {result.best_score:.2f}, Generations: {result.generations}")
print(result.best_code)

### 8. Adaptive evaluation with CodeModeAgent

The fixed-test-suite approach works well when you know exactly what to validate upfront. But it has a blind spot: the evaluation is only as good as the tests you wrote.

`CodeModeAgent` removes this constraint. Instead of a fixed test suite, the agent receives the task description and the candidate code, then **designs its own test cases** — including edge cases and boundary values it infers from the specification — and calls `run_tests_tool` to execute them.

| | Fixed test suite (sections 5–7) | Adaptive evaluation (this section) |
|--|--|--|
| **Test cases** | Hardcoded, passed as parameter | Agent generates from task description |
| **Task signature** | Requires `test_cases` + `fn_name` | Only `task_description` + `fn_name` |
| **Evaluation** | Same tests every generation | Agent varies tests based on what it notices |
| **Edge case coverage** | Limited to what you wrote | Agent infers from spec |

#### `run_tests_tool`: plain `async def`, not `@env.task`

`CodeModeAgent` routes tools into two paths based on their type:

- **`TaskTemplate`** (`@env.task`) → bridge calls `ref.aio()`, which schedules a new Flyte pod
- **Plain callable** → bridge calls it directly inside the Monty sandbox process

On devbox, `ref.aio()` from inside an already-running task fails silently — the pod scheduler returns `None` because nested task execution requires Union's `ReusePolicy` (warm container pool). See `examples/agents/codemode_durable_task_agent.py` for the Union pattern.

On devbox, `run_tests_tool` stays a plain `async def`. It runs in-process inside the Monty sandbox, which is sufficient since `evolve_program_adaptive` already runs in an isolated Flyte container.

In [ ]:
from flyte.ai.agents import CodeModeAgent

_image_adaptive = (
    flyte.Image.from_debian_base(name="evo-agent-adaptive", python_version=(3, 12))
    .with_pip_packages("anthropic>=0.25.0", "litellm", "pydantic-monty")
)

evo_env_adaptive = flyte.TaskEnvironment(
    name="evo_agent_adaptive",
    image=_image_adaptive,
    resources=flyte.Resources(cpu="1", memory="2Gi"),
    secrets=[
        flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY"),
    ],
    depends_on=[sandbox_environment],
)


# run_tests_tool reuses the same `eval_sandbox` defined above. It is a plain
# `async def` so CodeModeAgent's bridge calls it directly inside the Monty sandbox
# process (rather than scheduling a new Flyte pod), which on the devbox shares the
# running task's Flyte context and can therefore call eval_sandbox.run.aio().
async def run_tests_tool(code: str, fn_name: str, test_cases_json: str) -> str:
    """Run candidate code in the isolated eval sandbox; return JSON {score, passed, total, error_msg}.

    test_cases_json format: [{"inputs": [arg1, ...], "expected": <value>}, ...]
    Boolean values must use JSON booleans (true/false), not True/False.
    """
    import json as _j
    score, error_msg, passed, total = await eval_sandbox.run.aio(
        code=code, fn_name=fn_name, test_cases_json=test_cases_json
    )
    return _j.dumps({"score": score, "error_msg": error_msg, "passed": passed, "total": total})


eval_agent_adaptive = CodeModeAgent(
    tools=[run_tests_tool],
    model="claude-haiku-4-5",
    max_retries=2,
    system_prompt_prefix=(
        "You are an adaptive code evaluator. Given a task description and candidate Python function:\n"
        "1. Design thorough test cases including typical inputs, edge cases, and boundary values.\n"
        "2. Call run_tests_tool(code, fn_name, test_cases_json) where test_cases_json is a JSON array "
        "of {\"inputs\": [...], \"expected\": <value>} objects. "
        "Use JSON booleans (true/false), not Python booleans (True/False).\n"
        "3. Return {\"summary\": <the string result from run_tests_tool>}. "
        "The summary value must be the raw string returned by run_tests_tool — do not modify it."
    ),
)


async def _evaluate_adaptive(
    variant: ProgramVariant,
    task_description: str,
    fn_name: str,
) -> ProgramVariant:
    """Score a variant: CodeModeAgent designs the test cases, eval_sandbox runs them."""
    agent_result = await eval_agent_adaptive.run.aio(
        message=(
            f"Task: {task_description}\n\n"
            f"Function name: {fn_name}\n\n"
            f"Candidate code:\n```python\n{variant.code}\n```\n\n"
            "Design comprehensive test cases and call run_tests_tool to evaluate this function."
        ),
        history=[],
    )
    try:
        parsed = json.loads(agent_result.summary or "{}")
        score = float(parsed.get("score", 0.0))
        passed = parsed.get("passed", "?")
        total = parsed.get("total", "?")
        error_msg = parsed.get("error_msg", "")
        rationale = f"{passed}/{total} tests passed" + (f" — {error_msg}" if error_msg else "")
    except (json.JSONDecodeError, ValueError):
        score = 0.0
        rationale = agent_result.error or agent_result.summary or "evaluation failed"
    return ProgramVariant(
        code=variant.code,
        score=score,
        generation=variant.generation,
        mutation_notes=rationale,
    )


@flyte.trace
async def _critique_adaptive(variant: ProgramVariant, task_description: str) -> str:
    """Critique using the evaluation feedback stored in mutation_notes."""
    client = AsyncAnthropic()
    response = await client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=256,
        system="You are a code reviewer. Suggest ONE specific improvement for a Python function.",
        messages=[{
            "role": "user",
            "content": (
                f"Task: {task_description}\n"
                f"Current score: {variant.score:.2f}\n"
                f"Code:\n```python\n{variant.code}\n```\n"
                f"Test feedback: {variant.mutation_notes}\n\n"
                "In one sentence, what should the next mutation focus on?"
            ),
        }],
    )
    return response.content[0].text.strip()

In [ ]:
@evo_env_adaptive.task(
    retries=2,
    timeout=timedelta(minutes=20),
    cache=flyte.Cache(behavior="disable"),
    report=True,
)
async def evolve_program_adaptive(
    task_description: str,
    initial_code: str,
    fn_name: str,
    max_generations: int = 10,
    target_score: float = 1.0,
) -> EvoResult:
    """
    Evolutionary coding agent with adaptive evaluation.

    Unlike evolve_program, there are no hardcoded test cases — CodeModeAgent
    designs the test suite each generation based on the task description alone.
    The critique step also uses the agent's evaluation feedback rather than
    failing test cases, making mutation guidance richer and more targeted.
    """
    best = await _evaluate_adaptive(
        ProgramVariant(code=initial_code, generation=0),
        task_description,
        fn_name,
    )
    total_tried = 1
    report_rows: list[str] = []

    for gen in range(1, max_generations + 1):
        critique = await _critique_adaptive(best, task_description)
        mutant = await _mutate(best.code, task_description, critique)
        mutant = ProgramVariant(code=mutant.code, generation=gen, mutation_notes=mutant.mutation_notes)
        mutant = await _evaluate_adaptive(mutant, task_description, fn_name)
        total_tried += 1

        improved = mutant.score > best.score
        if improved:
            best = mutant

        color = "green" if improved else "gray"
        report_rows.append(
            f"<tr><td>{gen}</td>"
            f"<td style='color:{color}'>{mutant.score:.2f}</td>"
            f"<td>{best.score:.2f}</td>"
            f"<td>{_html_escape(critique[:80])}...</td>"
            f"<td>{_html_escape(best.mutation_notes[:60])}</td></tr>"
        )

        await flyte.report.replace.aio(
            "<!DOCTYPE html><html><body>"
            f"<h2>Adaptive: {_html_escape(task_description[:60])}</h2>"
            f"<p>Generation {gen} / {max_generations} | Best score: <strong>{best.score:.2f}</strong></p>"
            "<table border='1' cellpadding='4'>"
            "<tr><th>Gen</th><th>Mutant</th><th>Best</th><th>Critique</th><th>Test feedback</th></tr>"
            + "".join(report_rows)
            + "</table>"
            f"<h3>Current best code</h3><pre>{_html_escape(best.code)}</pre>"
            "</body></html>"
        )
        await flyte.report.flush.aio()

        if best.score >= target_score:
            break

    return EvoResult(
        task_description=task_description,
        best_code=best.code,
        best_score=best.score,
        generations=gen,
        total_variants_tried=total_tried,
        converged=best.score >= target_score,
    )

In [ ]:
run = await flyte.run.aio(
    evolve_program_adaptive,
    task_description=(
        "A function count_vowels(s: str) -> int that returns the number of vowels "
        "(a, e, i, o, u) in s, case-insensitive."
    ),
    initial_code="def count_vowels(s: str) -> int:\n    return 0  # placeholder",
    fn_name="count_vowels",
    max_generations=8,
    target_score=1.0,
)
result: EvoResult = run.outputs()[0]
print(f"Converged: {result.converged}")
print(f"Best score: {result.best_score:.2f} after {result.generations} generations ({result.total_variants_tried} variants)")
print(f"\nBest code:\n{result.best_code}")

### 9. Workflow sandbox with `orchestrate_local`

`flyte.sandbox.create` sandboxes a single code *snippet*. `orchestrate_local` sandboxes *control flow*: you hand it a Python string (which an LLM could generate) plus a whitelist of `tasks` it may call, and the **Monty runtime** executes it with strict restrictions (no imports, no `class`, no augmented assignment, last expression is the result). This is how you let a model drive dynamic loops/branching without giving it arbitrary execution.

In [ ]:
from flyte.sandbox import orchestrate_local


def score_code(code: str) -> float:
    """Trusted host-side scorer the sandboxed workflow may call (by __name__)."""
    ns: dict = {}
    try:
        exec(compile(code, "<cand>", "exec"), ns)
    except Exception:
        return 0.0
    fn = ns.get("is_even")
    if not callable(fn):
        return 0.0
    cases = [(2, True), (3, False), (0, True), (7, False)]
    passed = sum(1 for x, exp in cases if fn(x) == exp)
    return passed / len(cases)


# Monty dialect: no augmented assignment, no subscript assignment; last expr returns.
WORKFLOW_SRC = """
best_score = -1.0
best_code = ""
for candidate in candidates:
    s = score_code(candidate)
    if s > best_score:
        best_score = s
        best_code = candidate
best_code
"""

best_code = await orchestrate_local(
    WORKFLOW_SRC,
    inputs={"candidates": [
        "def is_even(n): return False",
        "def is_even(n): return n % 2 == 0",
    ]},
    tasks=[score_code],
)
print("orchestrate_local selected:\n", best_code)

## Scaling the pattern

The devbox runs each task in a fresh container. For production workloads with many short LLM calls, `ReusePolicy` eliminates cold-start overhead by keeping a pool of warm containers ready.

> **Note:** `ReusePolicy` is a Union-specific feature that requires a [Union deployment](https://www.union.ai/docs/v2/union/). It is not supported on the local devbox.

In [ ]:
# Requires a Union deployment — not supported on the local devbox
from datetime import timedelta

production_evo_agent = flyte.TaskEnvironment(
    name="evo_agent_prod",
    image=_image,
    resources=flyte.Resources(cpu="2", memory="4Gi"),
    secrets=[flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY")],
    reusable=flyte.ReusePolicy(
        replicas=(2, 8),
        concurrency=4,
        scaledown_ttl=timedelta(minutes=5),
        idle_ttl=timedelta(minutes=15),
    ),
)